# Laboratorio 03 â€” ConfiguraciÃ³n YAML para tu Pipeline

**Semana:** 04 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

DiseÃ±a un archivo de configuraciÃ³n YAML (`bronze_config.yml`) especÃ­fico para **tu dataset propio** y construye el notebook que lo lee y ejecuta la ingestiÃ³n Bronze de forma completamente declarativa, sin hard-coding de rutas o nombres de tabla.

## Parte 1 â€” DescripciÃ³n del dataset y diseÃ±o YAML

1. **Nombre, fuente y URL** del dataset.
2. **Entornos que tendrÃ¡ tu config:** Â¿DividirÃ¡s `dev` y `prod`? Â¿QuÃ© cambia entre ellos (rutas, nombres de tabla, opciones)?
3. **Estructura YAML:** Dibuja o describe la jerarquÃ­a de llaves que tendrÃ¡ tu archivo YAML antes de escribirlo.
4. **Campos del schema:** Si defines el esquema explÃ­cito en el YAML, lista las columnas con sus tipos.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Crear el archivo de configuraciÃ³n YAML

In [ ]:
import os

# Ruta del archivo de configuraciÃ³n en el driver de Databricks
CONFIG_DIR  = "/tmp/lab04_03"
CONFIG_PATH = f"{CONFIG_DIR}/bronze_config.yml"
os.makedirs(CONFIG_DIR, exist_ok=True)

# Escribe el contenido YAML de tu configuraciÃ³n
# Ajusta todos los valores para que correspondan a tu dataset real
config_yaml = """
entorno: dev

dev:
  fuentes:
    - nombre: mi_dataset
      descripcion: "Dataset principal del laboratorio"
      formato: csv
      ruta: /Volumes/workspace/default/week_4/tu_archivo.csv
      opciones:
        header: "true"
        inferSchema: "true"
        delimiter: ","
      destino:
        tabla: workspace.default.bronze_dev_mi_dataset
        modo: overwrite
      schema:
        - nombre: columna1
          tipo: string
        - nombre: columna2
          tipo: integer
        - nombre: columna3
          tipo: double
      calidad:
        max_pct_nulos_permitido: 30
        columnas_no_nulas:
          - columna1

prod:
  fuentes:
    - nombre: mi_dataset
      descripcion: "Dataset principal en producciÃ³n"
      formato: csv
      ruta: /Volumes/workspace/default/week_4/tu_archivo_prod.csv
      opciones:
        header: "true"
        inferSchema: "false"
        delimiter: ","
      destino:
        tabla: workspace.default.bronze_prod_mi_dataset
        modo: append
      calidad:
        max_pct_nulos_permitido: 10
        columnas_no_nulas:
          - columna1
          - columna2
"""

with open(CONFIG_PATH, "w") as f:
    f.write(config_yaml)

print(f"âœ“ ConfiguraciÃ³n YAML guardada en {CONFIG_PATH}")

## Parte 3 â€” Cargar y validar el YAML

In [ ]:
import yaml

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

entorno = config["entorno"]
env_cfg = config[entorno]

print(f"Entorno activo: {entorno}")
print(f"Fuentes configuradas: {len(env_cfg['fuentes'])}")
for fuente in env_cfg["fuentes"]:
    print(f"  - {fuente['nombre']}: {fuente['formato']} â†’ {fuente['destino']['tabla']}")

## Parte 4 â€” Perfil tÃ©cnico del dataset

In [ ]:
from pyspark.sql import functions as F

# Prueba de lectura antes de la ingestiÃ³n oficial
fuente_0 = env_cfg["fuentes"][0]

reader = spark.read.format(fuente_0["formato"])
for k, v in fuente_0.get("opciones", {}).items():
    reader = reader.option(k, v)

df_preview = reader.load(fuente_0["ruta"])
print(f"Preview: {df_preview.count():,} filas | {len(df_preview.columns)} columnas")
df_preview.printSchema()

In [ ]:
# Validar columnas no nulas declaradas en la config de calidad
cols_no_nulas = fuente_0.get("calidad", {}).get("columnas_no_nulas", [])
total = df_preview.count()

for col in cols_no_nulas:
    if col in df_preview.columns:
        nulos = df_preview.filter(F.col(col).isNull() | (F.col(col).cast("string") == "")).count()
        pct   = round(nulos * 100.0 / total, 1)
        estado = "âœ“" if nulos == 0 else "âœ—"
        print(f"{estado} {col}: {nulos:,} nulos ({pct}%)")
    else:
        print(f"âš  Columna '{col}' declarada en config pero no existe en el dataset")

**Calidad antes de ingesta:** Â¿Alguna columna definida como obligatoria en el YAML tiene nulos? Â¿QuÃ© harÃ­a el pipeline si fallara esta validaciÃ³n?

## Parte 5 â€” Ejecutar la ingestiÃ³n declarativa

In [ ]:
from datetime import datetime

resumen_ingestiones = []

for fuente in env_cfg["fuentes"]:
    nombre  = fuente["nombre"]
    formato = fuente["formato"]
    ruta    = fuente["ruta"]
    opciones = fuente.get("opciones", {})
    tabla   = fuente["destino"]["tabla"]
    modo    = fuente["destino"]["modo"]

    try:
        reader = spark.read.format(formato)
        for k, v in opciones.items():
            reader = reader.option(k, v)

        df = reader.load(ruta) \
            .withColumn("_ingest_ts",      F.current_timestamp()) \
            .withColumn("_source_file",    F.lit(ruta)) \
            .withColumn("_entorno",        F.lit(entorno))

        df.write.format("delta").mode(modo).saveAsTable(tabla)
        resumen_ingestiones.append({"fuente": nombre, "tabla": tabla, "filas": df.count(), "estado": "OK"})
        print(f"âœ“ {nombre} â†’ {tabla} ({df.count():,} filas, modo={modo})")

    except Exception as e:
        resumen_ingestiones.append({"fuente": nombre, "tabla": tabla, "filas": 0, "estado": f"ERROR: {e}"})
        print(f"âœ— {nombre}: {e}")

print("\n--- Resumen de ingestiones ---")
for r in resumen_ingestiones:
    print(r)

## Parte 6 â€” Verificar historial Delta y resultado

In [ ]:
tabla_ingesta = env_cfg["fuentes"][0]["destino"]["tabla"]

# Historial de versiones Delta
spark.sql(f"DESCRIBE HISTORY {tabla_ingesta}").select(
    "version", "timestamp", "operation", "operationParameters"
).show(5, truncate=False)

In [ ]:
# Muestra de los datos ingestados
spark.table(tabla_ingesta).select(
    "_ingest_ts", "_source_file", "_entorno"
).show(5, truncate=False)

## Parte 7 â€” Preguntas de negocio sobre los datos ingestados

In [ ]:
# Pregunta 1: Â¿CuÃ¡ntos registros hay y cuÃ¡ndo fue la Ãºltima ingestiÃ³n?
spark.sql(f"""
    SELECT
        COUNT(*) AS total_registros,
        MAX(_ingest_ts) AS ultima_ingestiÃ³n,
        _entorno
    FROM {tabla_ingesta}
    GROUP BY _entorno
""").show()

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2: anÃ¡lisis de negocio libre sobre los datos ya en Bronze
spark.sql(f"""
    -- Escribe tu consulta de negocio aquÃ­
    SELECT * FROM {tabla_ingesta} LIMIT 10
""").show(truncate=False)

**ConclusiÃ³n pregunta 2:**

## Parte 8 â€” ReflexiÃ³n final

1. Â¿Por quÃ© `yaml.safe_load()` es mÃ¡s seguro que `yaml.load()`?
2. Â¿QuÃ© sucedeÃ­a si cambias `entorno: prod` en el YAML y vuelves a ejecutar el notebook? Â¿Hay algÃºn riesgo?
3. Â¿QuÃ© pasa si el campo `calidad.columnas_no_nulas` referencia una columna que no existe en el archivo CSV? Â¿CÃ³mo lo detectarÃ­as antes de la ingestiÃ³n?
4. Â¿CÃ³mo refactorizarÃ­as este notebook para que pueda leer archivos YAML almacenados en DBFS o en un volumen de UC en lugar de `/tmp`?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_04/laboratorios/lab_03_yaml_config.ipynb semana_04/laboratorios/<tu-nombre>/lab_03_yaml_config.ipynb

git add semana_04/laboratorios/<tu-nombre>/lab_03_yaml_config.ipynb
git commit -m "lab: semana04 lab03 YAML config bronze pipeline <nombre-dataset> - <tu-nombre>"
git push origin develop
```